In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:1','cuda:6'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

('cuda:1', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78941)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 68805)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 62805)

# #### Device() ####
# device = cuda:3

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:3)
# edge_attr                (32464, 16)              Tensor (cuda:3)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:3)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

---

In [2]:
from modules.layers import AttentionSetPooling
from modules.model import LatentClassifier
from modules.norm import ZlogNorm
from modules.train import ClassifTrainer, Experiment, grid

import torch.nn as nn
from functools import partial

In [3]:
model_template = partial(
    LatentClassifier,
    dataset=dataset,
    method='node',
    
    # layers
    norm_class=ZlogNorm,
    encoder_class=nn.Linear,
    pooling_class=AttentionSetPooling,
    mlp=False,

    # layer params
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,
)

model_grid = grid(
    model_template,
    embed_dim=[16,32,64,128,256],
    hidden_dims=[0,1,2,3,4,5,6],
)

In [4]:
trainer_template = partial(
    ClassifTrainer,
    loss_class=nn.CrossEntropyLoss,
    report_metrics=['loss','accuracy'],
    weight_method='balanced'
)

trainer_grid = grid(
    trainer_template,
    lr=[1e-1,1e-2,1e-3,1e-4,1e-5],
)

In [5]:
expt = Experiment(
    num_trials=1,
    num_epochs=50,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

175


In [6]:
expt.run_experiment(
    'classif_grid_1t_175configs',
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/175 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]